<a href="https://colab.research.google.com/github/ramiyer-0/Clinical-Document-QA-Agent/blob/main/clinical_qa_agent_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clinical document QA agent

A RAG - based QA agent that answers question over clinical documents.

**What we used:**
- `pypdf` — for text extraction
- `sentence-transformers` — for embedding
- `faiss-cpu` — as the vector databse
- `transformers` — as the LLM


In [43]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate


## Step 1 — Uploading and Extraction

After running this code block, you can add a clinical document as PDF. We use pypdf to extract text from the pdf.


In [44]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

reader = PdfReader(pdf_path)
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"Extracted {len(full_text)} characters from {len(reader.pages)} pages")
print(full_text[:500])  # preview


Saving SampleClinicalDocument.pdf to SampleClinicalDocument (2).pdf
Extracted 14806 characters from 6 pages
Sample Written History and Physical Examination 
 
History and Physical Examination     Comments 
 
Patient Name: Rogers, Pamela 
Date: 6/2/04 
 
Referral Source: Emergency Department 
 
Data Source: Patient 
 
Chief Complaint & ID:  Ms. Rogers is a 56 y/o WF   Define the r eason for the patient’s visit as who has been 
having chest pains for the last week.     specifically as possible.  
      
 
History of Present Illness 
 
This is the first admission for this 56 year old woman,   Convey the 


## Step 2 — Chunking

We split the document into 500-character chunks that overlap by 50 characters.

In [45]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Simple character-based chunker with overlap between consecutive chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # step back by overlap so chunks share some text
    return [c.strip() for c in chunks if c.strip()]

chunks = chunk_text(full_text, chunk_size=500, overlap=50)
print(f"Created {len(chunks)} chunks")
print(chunks[0])


Created 33 chunks
Sample Written History and Physical Examination 
 
History and Physical Examination     Comments 
 
Patient Name: Rogers, Pamela 
Date: 6/2/04 
 
Referral Source: Emergency Department 
 
Data Source: Patient 
 
Chief Complaint & ID:  Ms. Rogers is a 56 y/o WF   Define the r eason for the patient’s visit as who has been 
having chest pains for the last week.     specifically as possible.  
      
 
History of Present Illness 
 
This is the first admission for this 56 year old woman,   Convey the


## Step 3 — Embedding

Embed the chunks and build a FAISS index.


In [46]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode all chunks into vectors, normalized so we can use cosine similarity
chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # IP = inner product = cosine similarity on normalized vectors
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built with 33 vectors of dimension 384


## Step 4 — Retrieval function

Given a question, embed it the same way, then ask FAISS for the top-k most similar chunks.


In [47]:
def retrieve(question, k=5):
    query_vector = embedder.encode([question], normalize_embeddings=True).astype("float32")
    similarities, indices = index.search(query_vector, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks, similarities[0]

# quick test
test_question = "Does the patient have bleeding?"
retrieved_chunks, scores = retrieve(test_question, k=3)
for chunk, score in zip(retrieved_chunks, scores):
    print(f"score={score:.3f} | {chunk[:150]}...\n")


score=0.493 | polyuria, hematuria, or 
vaginal bleeding. 
 
Musculoskeletal: 
She complains of lower back pain, aching in quality, 
approximately once every week af...

score=0.379 | the patient to the telemetry floor. 
 
2. Start platelet inhibitors, such as aspirin to decrease the risk of 
myocardial infarction; start nitrates to...

score=0.376 | odes in the cervical, supraclavicular, axillary 
or inguinal areas. 
 
Genital/Rectal: 
Normal rectal sphincter tone;  no rectal masses or   Always in...



## Step 5 — Answer Generation using LLM

We use `Qwen2.5-1.5B-Instruct`, which is strong enough to follow the
"answer only from context" instruction reliably at this size.

In [48]:
from transformers import pipeline
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=device,
)

def answer_question(question, k=5, max_new_tokens=200):
    retrieved_chunks, scores = retrieve(question, k=k)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say you don't know — do not guess.

Context:
{context}

Question: {question}
Answer:"""

    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_new_tokens, do_sample=False)
    response = output[0]["generated_text"][-1]["content"]
    return response, retrieved_chunks


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Step 6 — Query & Answer


In [49]:
answer, sources = answer_question("Does she generally practice a healthy lifestyle?")
print("ANSWER:\n", answer)
print("\n---\nRetrieved from:")
for s in sources:
    print("-", s[:120], "...")


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:
 Yes, she generally practices a healthy lifestyle. The context mentions that she has been prescribed a low-fat diet to manage her condition, indicating that she is taking steps to maintain a healthier lifestyle. Additionally, she reports experiencing lower back pain due to gardening activities, suggesting she engages in some form of regular physical activity despite the occasional discomfort. However, without specific information about her dietary habits or overall health regimen, we cannot definitively state whether she follows a perfect healthy lifestyle.

---
Retrieved from:
- on.  She was diagnosed with 
HTN 3 years ago,  
 
She does not smoke nor does she have diabetes.    Relevant risk factor ...
- tarted on an appropriate exercise and 
weight loss program, including a low-fat diet.  If her cholesterol 
is elevated,  ...
- polyuria, hematuria, or 
vaginal bleeding. 
 
Musculoskeletal: 
She complains of lower back pain, aching in quality, 
ap ...
- n supports the 
theory t

## Step 7 — Evaluation Framework

50 queries and the keywords we expect to be in their retrieved chunks to evaluate retrieval precision.


In [50]:
test_cases = [
    # --- Demographics & chief complaint / HPI (15) ---
    {"question": "What is the patient's name?", "expected_keywords": ["rogers", "pamela"]},
    {"question": "How old is the patient?", "expected_keywords": ["56"]},
    {"question": "What is the patient's sex?", "expected_keywords": ["woman", "wf", "female"]},
    {"question": "What was the referral source for this admission?", "expected_keywords": ["emergency department"]},
    {"question": "What is the patient's chief complaint?", "expected_keywords": ["chest pain"]},
    {"question": "How long had the patient been having chest pain before this admission?", "expected_keywords": ["week"]},
    {"question": "Where was the patient when her first episode of chest pain began?", "expected_keywords": ["garden"]},
    {"question": "How does the patient describe the character of her chest pain?", "expected_keywords": ["dull", "aching"]},
    {"question": "Where did the chest pain radiate to?", "expected_keywords": ["neck"]},
    {"question": "How long did the first episode of chest pain last?", "expected_keywords": ["5 to 10 minutes"]},
    {"question": "What relieved the first episode of chest pain?", "expected_keywords": ["rest", "cool"]},
    {"question": "How many additional episodes of pain occurred after the first one?", "expected_keywords": ["2", "two"]},
    {"question": "What was the patient doing during the episode of pain 3 days ago?", "expected_keywords": ["walking", "dog"]},
    {"question": "How long did the episode of pain 3 days ago last?", "expected_keywords": ["15 minute"]},
    {"question": "What happened during tonight's episode that prompted her ED visit?", "expected_keywords": ["sleep", "awaken", "30 minutes"]},

    # --- Past medical / social / habits (10) ---
    {"question": "Does the patient have diabetes?", "expected_keywords": ["diabetes"]},
    {"question": "Does the patient smoke?", "expected_keywords": ["smoke"]},
    {"question": "When was the patient diagnosed with hypertension?", "expected_keywords": ["3 years", "hypertension"]},
    {"question": "What surgery did the patient have in 1994?", "expected_keywords": ["hysterectomy", "oophorectomy"]},
    {"question": "What surgery did the patient have in 1998?", "expected_keywords": ["bunionectomy"]},
    {"question": "What medication treated her peptic ulcer disease in 1990?", "expected_keywords": ["cimetidine"]},
    {"question": "What medication allergy does the patient have?", "expected_keywords": ["penicillin"]},
    {"question": "What reaction did she have to her drug allergy?", "expected_keywords": ["rash", "hives"]},
    {"question": "What is the patient's alcohol use like?", "expected_keywords": ["beer", "wine"]},
    {"question": "What over-the-counter medication does the patient take?", "expected_keywords": ["ibuprofen", "advil"]},

    # --- Family history & review of systems (5) ---
    {"question": "How did the patient's father die?", "expected_keywords": ["heart attack"]},
    {"question": "What is the health status of the patient's mother?", "expected_keywords": ["79", "alive"]},
    {"question": "Is there a relevant family history related to the chest pain?", "expected_keywords": ["family history", "cad"]},
    {"question": "What gastrointestinal complaint does the patient report?", "expected_keywords": ["epigastric", "burning"]},
    {"question": "What musculoskeletal complaint does the patient have?", "expected_keywords": ["back pain"]},

    # --- Physical exam (10) ---
    {"question": "What was the patient's blood pressure on exam?", "expected_keywords": ["168/98"]},
    {"question": "What was the patient's pulse on exam?", "expected_keywords": ["90"]},
    {"question": "What abnormal heart sounds were heard on exam?", "expected_keywords": ["murmur", "heart sound"]},
    {"question": "Where was the systolic murmur heard best?", "expected_keywords": ["intercostal"]},
    {"question": "What abnormal finding was noted in the abdomen?", "expected_keywords": ["bruit"]},
    {"question": "What was the patient's liver span on exam?", "expected_keywords": ["8 cm"]},
    {"question": "What lung finding was noted bilaterally?", "expected_keywords": ["crackles"]},
    {"question": "What was the patient's jugular venous pressure?", "expected_keywords": ["jugular", "8 cm"]},
    {"question": "What was found on examination of the peripheral pulses?", "expected_keywords": ["pulses", "normal"]},
    {"question": "What were the findings on the neurological exam?", "expected_keywords": ["cranial nerves", "normal"]},

    # --- Assessment, differential diagnosis & plan (10) ---
    {"question": "What is the most likely diagnosis for the patient's chest pain?", "expected_keywords": ["angina", "ischemic"]},
    {"question": "Why is GERD considered a less likely cause of her chest pain?", "expected_keywords": ["gerd", "exertion"]},
    {"question": "What might the patient's dyspnea, third heart sound, and crackles together suggest?", "expected_keywords": ["congestive heart failure", "left ventricular"]},
    {"question": "What is suspected as a possible cause of her recent-onset hypertension?", "expected_keywords": ["renal artery", "renovascular"]},
    {"question": "What valvular disease is being considered due to the murmur?", "expected_keywords": ["aortic stenosis"]},
    {"question": "What medication is recommended to reduce the risk of myocardial infarction?", "expected_keywords": ["aspirin"]},
    {"question": "What test is scheduled to evaluate her coronary arteries?", "expected_keywords": ["catheterization", "cath"]},
    {"question": "What class of medication might be needed if her cholesterol is elevated?", "expected_keywords": ["hmg", "co-reductase"]},
    {"question": "What lab tests are planned to assess kidney function?", "expected_keywords": ["bun", "creatinine"]},
    {"question": "What unit is the patient being admitted to?", "expected_keywords": ["telemetry"]},
]

def evaluate_retrieval(test_cases, k=5):
    results = []
    for case in test_cases:
        retrieved_chunks, _ = retrieve(case["question"], k=k)
        retrieved_text = " ".join(retrieved_chunks).lower()
        hit = any(kw.lower() in retrieved_text for kw in case["expected_keywords"])
        results.append({"question": case["question"], "retrieval_hit": hit})
    return results

results = evaluate_retrieval(test_cases)
precision = sum(r["retrieval_hit"] for r in results) / len(results)
print(f"Retrieval precision (proxy): {precision:.0%}")
for r in results:
    print(r)


Retrieval precision (proxy): 94%
{'question': "What is the patient's name?", 'retrieval_hit': True}
{'question': 'How old is the patient?', 'retrieval_hit': True}
{'question': "What is the patient's sex?", 'retrieval_hit': True}
{'question': 'What was the referral source for this admission?', 'retrieval_hit': True}
{'question': "What is the patient's chief complaint?", 'retrieval_hit': True}
{'question': 'How long had the patient been having chest pain before this admission?', 'retrieval_hit': True}
{'question': 'Where was the patient when her first episode of chest pain began?', 'retrieval_hit': True}
{'question': 'How does the patient describe the character of her chest pain?', 'retrieval_hit': True}
{'question': 'Where did the chest pain radiate to?', 'retrieval_hit': True}
{'question': 'How long did the first episode of chest pain last?', 'retrieval_hit': True}
{'question': 'What relieved the first episode of chest pain?', 'retrieval_hit': True}
{'question': 'How many additional ep

## Step 8 — Chunk Size Optimization

Improving retrieval precision by altering the chunk size.


In [51]:
def build_index(text, chunk_size, overlap=50):
    new_chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
    new_embeddings = embedder.encode(new_chunks, normalize_embeddings=True).astype("float32")
    new_index = faiss.IndexFlatIP(new_embeddings.shape[1])
    new_index.add(new_embeddings)
    return new_chunks, new_index

for size in [300, 500, 800]:
    trial_chunks, trial_index = build_index(full_text, chunk_size=size)

    def trial_retrieve(question, k=5):
        qv = embedder.encode([question], normalize_embeddings=True).astype("float32")
        _, idx = trial_index.search(qv, k)
        return [trial_chunks[i] for i in idx[0]]

    hits = 0
    for case in test_cases:
        retrieved = " ".join(trial_retrieve(case["question"])).lower()
        if any(kw.lower() in retrieved for kw in case["expected_keywords"]):
            hits += 1
    print(f"chunk_size={size}: precision={hits/len(test_cases):.0%}")


chunk_size=300: precision=88%
chunk_size=500: precision=94%
chunk_size=800: precision=98%


## Retrieval Precision Improvement
Retrieval Precision improved by 4% by changing the chunk size from 500 to 800 characters.